In [3]:
require 'csv'
require 'net/http'
require 'json'
require 'uri'

# Collect unique (CUI, UMLS name) pairs from all six phenotype TSVs
# quote_char: "\x00" disables quote parsing — PROVENANCE column contains
# JSON arrays like ["123_fullText_1"] that trip up the default CSV parser
tsv_files = Dir.glob('raw-data/*Phenotype*.tsv')
puts "Found files: #{tsv_files.map { |f| File.basename(f) }.join(', ')}"

cui_map = {}
tsv_files.each do |f|
  CSV.foreach(f, col_sep: "\t", headers: true, quote_char: "\x00") do |row|
    cui  = row['Phenotype_id']&.strip
    name = row['Phenotype']&.strip
    cui_map[cui] ||= name if cui
  end
end
puts "Unique CUIs: #{cui_map.size}"

# Search Monarch for a UMLS CUI restricted to PhenotypicFeature category.
# Returns [:ok, hp_id, hp_name], [:not_found, '', ''], or [:xref_mismatch, candidate_id, candidate_name]
def search_hpo(cui)
  uri = URI('https://api-v3.monarchinitiative.org/v3/api/search')
  uri.query = URI.encode_www_form(q: "UMLS:#{cui}", category: 'biolink:PhenotypicFeature', limit: 1)
  res = Net::HTTP.get_response(uri)
  return [:not_found, '', ''] unless res.code == '200'
  data = JSON.parse(res.body)
  return [:not_found, '', ''] if data['total'].to_i.zero?
  item = data['items'].first
  xrefs = Array(item['xref'])
  return [:xref_mismatch, item['id'], item['name'].to_s] unless xrefs.include?("UMLS:#{cui}")
  [:ok, item['id'], item['name'].to_s]
rescue => e
  [:"error_#{e.message.gsub(/\s+/, '_')[0,40]}", '', '']
end

mapped   = []
errors   = []

cui_map.each_with_index do |(cui, umls_name), i|
  puts "  #{i}/#{cui_map.size} ..." if (i % 50).zero?
  status, hpo_id, hpo_name = search_hpo(cui)
  if status == :ok
    mapped << { CUI: cui, UMLS_Name: umls_name, HPO: hpo_id, HPO_Name: hpo_name }
  else
    errors << { CUI: cui, UMLS_Name: umls_name, Reason: status.to_s,
                Candidate_HPO: hpo_id, Candidate_HPO_Name: hpo_name }
  end
  sleep 0.1
end

puts "\nMapped:  #{mapped.size} / #{cui_map.size} (#{'%.1f' % (100.0 * mapped.size / cui_map.size)}%)"
puts "Errors:  #{errors.size}"
puts "  not_found:     #{errors.count { |e| e[:Reason] == 'not_found' }}"
puts "  xref_mismatch: #{errors.count { |e| e[:Reason] == 'xref_mismatch' }}"

CSV.open('maps/cui_hpo_lookup.tsv', 'w', col_sep: "\t") do |csv|
  csv << %w[CUI UMLS_Name HPO HPO_Name]
  mapped.each { |r| csv << [r[:CUI], r[:UMLS_Name], r[:HPO], r[:HPO_Name]] }
end

File.open('maps/2026-phenotype-mapping-errors.txt', 'w') do |f|
  errors.each do |e|
    candidate = e[:Candidate_HPO].empty? ? '' : "  (candidate: #{e[:Candidate_HPO]} #{e[:Candidate_HPO_Name]})"
    f.puts "#{e[:Reason]}\tUMLS:#{e[:CUI]}\t#{e[:UMLS_Name]}#{candidate}"
  end
end

puts "\nWritten: maps/cui_hpo_lookup.tsv  |  maps/2026-phenotype-mapping-errors.txt"
mapped.first(5)

Found files: Disease-Phenotype triples.tsv, Drug-Phenotype triples.tsv, Gene-Phenotype triples.tsv, Phenotype-Disease triples.tsv, Phenotype-Drug triples.tsv, Phenotype-Gene triples.tsv
Unique CUIs: 1166
  0/1166 ...
  50/1166 ...
  100/1166 ...
  150/1166 ...
  200/1166 ...
  250/1166 ...
  300/1166 ...
  350/1166 ...
  400/1166 ...
  450/1166 ...
  500/1166 ...
  550/1166 ...
  600/1166 ...
  650/1166 ...
  700/1166 ...
  750/1166 ...
  800/1166 ...
  850/1166 ...
  900/1166 ...
  950/1166 ...
  1000/1166 ...
  1050/1166 ...
  1100/1166 ...
  1150/1166 ...

Mapped:  238 / 1166 (20.4%)
Errors:  928
  not_found:     928
  xref_mismatch: 0

Written: maps/cui_hpo_lookup.tsv  |  maps/2026-phenotype-mapping-errors.txt


[{:CUI=>"C0026821", :UMLS_Name=>"Muscle Cramp", :HPO=>"HP:0003394", :HPO_Name=>"Muscle spasm"}, {:CUI=>"C1864985", :UMLS_Name=>"Progressive disorder", :HPO=>"HP:0003676", :HPO_Name=>"Progressive"}, {:CUI=>"C1850496", :UMLS_Name=>"Neuronal loss", :HPO=>"HP:0002529", :HPO_Name=>"Neuronal loss in central nervous system"}, {:CUI=>"C0007758", :UMLS_Name=>"Cerebellar Ataxia", :HPO=>"HP:0001251", :HPO_Name=>"Ataxia"}, {:CUI=>"C0234133", :UMLS_Name=>"Extrapyramidal sign", :HPO=>"HP:0002071", :HPO_Name=>"Abnormality of extrapyramidal motor function"}]